## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [ ]:
import seaborn as sns

titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
263,0,1,male,40.0,0,0,0.0000,S,First,man,True,B,Southampton,no,True
688,0,3,male,18.0,0,0,7.7958,S,Third,man,True,NaN,Southampton,no,True
530,1,2,female,2.0,1,1,26.0000,S,Second,child,False,NaN,Southampton,yes,False
117,0,2,male,29.0,1,0,21.0000,S,Second,man,True,NaN,Southampton,no,False
584,0,3,male,NaN,0,0,8.7125,C,Third,man,True,NaN,Cherbourg,no,True


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [10]:
print(titanic_data.isna().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [11]:
titanic_data_changed = titanic_data.dropna(axis="columns", thresh=titanic_data.shape[0] // 2)
titanic_data_changed = titanic_data_changed.dropna(
    axis="index", thresh=titanic_data_changed.shape[1] // 2
)
print(titanic_data_changed.isna().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
embark_town      2
alive            0
alone            0
dtype: int64


### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [13]:
categories = ("man", "woman", "child")

for i in categories:
    median = round(titanic_data_changed[titanic_data_changed["who"] == i]["age"].median())
    titanic_data_changed.loc[titanic_data_changed["who"] == i, "age"] = titanic_data_changed[
        titanic_data_changed["who"] == i
    ]["age"].fillna(median)
print(titanic_data_changed.isna().sum())

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       2
class          0
who            0
adult_male     0
embark_town    2
alive          0
alone          0
dtype: int64


### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [22]:
titanic_data_changed = titanic_data_changed.dropna(thresh=titanic_data_changed.shape[1] - 1)
print(titanic_data_changed.isna().sum())

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64


### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [24]:
max_city = titanic_data_changed["embark_town"].value_counts()
max_city.index[0]

'Southampton'

### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [30]:
parts = titanic_data_changed["survived"].value_counts(normalize=True)
proprtion = parts[1] * 100
round(proprtion, 2)

np.float64(38.25)

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [ ]:
survivors_count = titanic_data_changed[titanic_data_changed["survived"] == 1][
    "embark_town"
].value_counts()
survivors_count

embark_town
Southampton    217
Cherbourg       93
Queenstown      30
Name: count, dtype: int64

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [35]:
survivors_class = titanic_data_changed[titanic_data_changed["survived"] == 1]["class"].value_counts(
    normalize=True
)
survivors_class *= 100
round(survivors_class, 2)

class
First     39.41
Third     35.00
Second    25.59
Name: proportion, dtype: float64

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [39]:
wealthy = titanic_data_changed[titanic_data_changed["fare"] >= 100]
wealthy_survivor = wealthy["survived"].value_counts(normalize=True)
wealthy_survivor[1] *= 100
round(wealthy_survivor[1], 2)

np.float64(73.58)

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [42]:
children = titanic_data_changed[titanic_data_changed["who"] == "child"]
solo_children = children[(children["parch"] == 0) & (children["sibsp"] == 0)]
solo_children.shape[0]

6

Какие выводы вы можете сделать о выживших пассажирах Титаника? 